# Cryptocurrency Data Cleaning and Quality Validation

This notebook loads the hourly Binance Spot OHLCV datasets for 10 cryptocurrency pairs from January 2023 to December 2025.

The objectives are to:

- validate the downloaded files;
- check data types and missing values;
- identify duplicate and missing timestamps;
- verify OHLC price consistency;
- inspect invalid prices and volumes;
- create cleaned datasets for later analysis and backtesting.

In [5]:
from pathlib import Path

import numpy as np
import pandas as pd

In [7]:
symbols = [
    "BTCUSDT",
    "ETHUSDT",
    "BNBUSDT",
    "XRPUSDT",
    "ADAUSDT",
    "DOGEUSDT",
    "SOLUSDT",
    "LTCUSDT",
    "LINKUSDT",
    "AVAXUSDT",
]

interval = "1h"

raw_data_dir = Path("data/raw")
processed_data_dir = Path("data/processed")

processed_data_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("Raw data folder:")
print(raw_data_dir.resolve())

print("\nProcessed data folder:")
print(processed_data_dir.resolve())

Raw data folder:
/Users/nimo/crypto-statistical-arbitrage/data/raw

Processed data folder:
/Users/nimo/crypto-statistical-arbitrage/data/processed


In [9]:
all_data = {}

for symbol in symbols:
    file_path = (
        raw_data_dir
        / f"{symbol}_{interval}_2023_2025.parquet"
    )

    df = pd.read_parquet(file_path)

    all_data[symbol] = df

    print(
        f"{symbol}: "
        f"{len(df):,} rows, "
        f"{df['open_time'].min()} to "
        f"{df['open_time'].max()}"
    )

BTCUSDT: 26,303 rows, 2023-01-01 00:00:00+00:00 to 2025-12-31 23:00:00+00:00
ETHUSDT: 26,303 rows, 2023-01-01 00:00:00+00:00 to 2025-12-31 23:00:00+00:00
BNBUSDT: 26,303 rows, 2023-01-01 00:00:00+00:00 to 2025-12-31 23:00:00+00:00
XRPUSDT: 26,303 rows, 2023-01-01 00:00:00+00:00 to 2025-12-31 23:00:00+00:00
ADAUSDT: 26,303 rows, 2023-01-01 00:00:00+00:00 to 2025-12-31 23:00:00+00:00
DOGEUSDT: 26,303 rows, 2023-01-01 00:00:00+00:00 to 2025-12-31 23:00:00+00:00
SOLUSDT: 26,303 rows, 2023-01-01 00:00:00+00:00 to 2025-12-31 23:00:00+00:00
LTCUSDT: 26,303 rows, 2023-01-01 00:00:00+00:00 to 2025-12-31 23:00:00+00:00
LINKUSDT: 26,303 rows, 2023-01-01 00:00:00+00:00 to 2025-12-31 23:00:00+00:00
AVAXUSDT: 26,303 rows, 2023-01-01 00:00:00+00:00 to 2025-12-31 23:00:00+00:00


In [11]:
btc_df = all_data["BTCUSDT"]

btc_df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,ignore,symbol
0,2023-01-01 00:00:00+00:00,16541.77,16545.70,16508.39,16529.67,4364.83570,2023-01-01 00:59:59.999000+00:00,7.214629e+07,149854,2179.94772,3.603235e+07,0,BTCUSDT
1,2023-01-01 01:00:00+00:00,16529.59,16556.80,16525.78,16551.47,3590.06669,2023-01-01 01:59:59.999000+00:00,5.937676e+07,126556,1730.24901,2.861742e+07,0,BTCUSDT
2,2023-01-01 02:00:00+00:00,16551.47,16559.77,16538.14,16548.19,3318.84038,2023-01-01 02:59:59.999000+00:00,5.491945e+07,115398,1611.12302,2.666087e+07,0,BTCUSDT
3,2023-01-01 03:00:00+00:00,16548.19,16548.19,16518.21,16533.04,4242.08050,2023-01-01 03:59:59.999000+00:00,7.012254e+07,137724,2096.09287,3.464904e+07,0,BTCUSDT
4,2023-01-01 04:00:00+00:00,16533.04,16535.97,16511.92,16521.85,4285.00909,2023-01-01 04:59:59.999000+00:00,7.080264e+07,129535,2188.40175,3.615982e+07,0,BTCUSDT


In [13]:
btc_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26303 entries, 0 to 26302
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   open_time               26303 non-null  datetime64[ns, UTC]
 1   open                    26303 non-null  float64            
 2   high                    26303 non-null  float64            
 3   low                     26303 non-null  float64            
 4   close                   26303 non-null  float64            
 5   volume                  26303 non-null  float64            
 6   close_time              26303 non-null  datetime64[ns, UTC]
 7   quote_asset_volume      26303 non-null  float64            
 8   number_of_trades        26303 non-null  int64              
 9   taker_buy_base_volume   26303 non-null  float64            
 10  taker_buy_quote_volume  26303 non-null  float64            
 11  ignore                  26303 non-null  i

-- Loaded data for 10 cryptocurrencies

-- Verified that all files exist

-- Confirmed the date range

-- Inspected data samples

-- Checked the data type of the BTC column

-- Confirmed that BTC has no null values

 ## We start with cleaning and merging.



In [19]:
cleaned_data = {}

for symbol, df in all_data.items():

    clean_df = (
        df.copy()
        .sort_values("open_time") # sort by time 
        .drop_duplicates(       
            subset=["symbol", "open_time"]
        )                   # drop duplicate
        .reset_index(drop=True)
    )

    cleaned_data[symbol] = clean_df

    print(
        f"{symbol}: "
        f"{len(df):,} rows before cleaning, "
        f"{len(clean_df):,} rows after cleaning"
    )

BTCUSDT: 26,303 rows before cleaning, 26,303 rows after cleaning
ETHUSDT: 26,303 rows before cleaning, 26,303 rows after cleaning
BNBUSDT: 26,303 rows before cleaning, 26,303 rows after cleaning
XRPUSDT: 26,303 rows before cleaning, 26,303 rows after cleaning
ADAUSDT: 26,303 rows before cleaning, 26,303 rows after cleaning
DOGEUSDT: 26,303 rows before cleaning, 26,303 rows after cleaning
SOLUSDT: 26,303 rows before cleaning, 26,303 rows after cleaning
LTCUSDT: 26,303 rows before cleaning, 26,303 rows after cleaning
LINKUSDT: 26,303 rows before cleaning, 26,303 rows after cleaning
AVAXUSDT: 26,303 rows before cleaning, 26,303 rows after cleaning


In [21]:
columns_to_keep = [
    "open_time",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "quote_asset_volume",
    "number_of_trades",
    "taker_buy_base_volume",
    "taker_buy_quote_volume",
    "symbol",
] # Drop unnecessary columns: close time, ignore

In [23]:
for symbol in cleaned_data:

    cleaned_data[symbol] = (
        cleaned_data[symbol][columns_to_keep]
        .copy()
    )

In [36]:
combined_df = pd.concat(
    cleaned_data.values(),
    ignore_index=True,
) #Merge data for all 10 cryptocurrencies

combined_df = (
    combined_df
    .sort_values(
        ["open_time", "symbol"]
    )   # sort from open time and symbol
    .reset_index(drop=True)
)

combined_df.shape

(263030, 11)

In [27]:
## storage

In [38]:
processed_data_dir = Path("data/processed")

processed_data_dir.mkdir(
    parents=True,
    exist_ok=True,
)

In [40]:
combined_output_path = (
    processed_data_dir
    / "crypto_10_assets_1h_clean.parquet"
)

combined_df.to_parquet(
    combined_output_path,
    index=False,
)

print("Combined cleaned dataset saved to:")
print(combined_output_path.resolve())

Combined cleaned dataset saved to:
/Users/nimo/crypto-statistical-arbitrage/data/processed/crypto_10_assets_1h_clean.parquet


In [42]:
for symbol, df in cleaned_data.items():

    output_path = (
        processed_data_dir
        / f"{symbol}_{interval}_clean.parquet"
    )

    df.to_parquet(
        output_path,
        index=False,
    )

print("Individual cleaned datasets saved.")

Individual cleaned datasets saved.


## Cleaning Summary

The 10 hourly cryptocurrency datasets were successfully loaded, sorted chronologically and deduplicated by symbol and timestamp.

The cleaned datasets were saved both individually and as a combined panel dataset for exploratory analysis and backtesting.

No synthetic price interpolation was applied.

In [49]:
## data cleaning
##- loaded the individual hourly datasets, 
#- sorted chronologically by asset,
#- deduplicated using the symbol–timestamp combination.
#- Research-relevant OHLCV and trading-activity fields were retained,
#- the ten assets were consolidated into a single panel dataset for cross-sectional analysis.